## Regularization with SciKit-Learn

In our previous steps, we created a new polynomial feature set and applied standard linear regression. However, we can make more informed choices about our model by incorporating regularization techniques.

Regularization aims to minimize the Residual Sum of Squares (RSS) while also introducing a penalty factor. This penalty factor discourages models with excessively large coefficients. In some cases, regularization methods can even set certain feature coefficients to zero, effectively excluding them from the model.

Let's delve into two common regularization methods: Ridge Regression and Lasso. We'll apply these techniques in combination with our polynomial feature set. It's important to note that regularization is particularly beneficial when dealing with datasets that have a larger number of features, as it helps prevent overfitting and selects the most relevant features.

By exploring Ridge Regression and Lasso regularization in the context of our polynomial feature set, we aim to find a more robust and optimized model that balances the fit to the data with complexity control.


## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Data and Setup

In [6]:
df = pd.read_csv("Advertising.csv")
X = df.drop('sales',axis=1)
y = df['sales']

### Polynomial Conversion

In [7]:
from sklearn.preprocessing import PolynomialFeatures

In [8]:
polynomial_converter = PolynomialFeatures(degree=3,include_bias=False)

In [9]:
poly_features = polynomial_converter.fit_transform(X)

### Train | Test Split

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(poly_features, y, test_size=0.3, random_state=101)

## Scaling the Data

In our current dataset, all the values are in the same order of magnitude (in the thousands of dollars spent). However, this level of uniformity may not hold true for many datasets. When working with regularized models, it's crucial to standardize or scale the features, as these models sum coefficients together.

**Why Scaling Matters:**
- Scaling ensures that all features have a similar influence on the regularization process, regardless of their original scales.
- Regularization methods like Ridge and Lasso operate based on the magnitude of coefficients, making feature scaling essential.
- Standardizing the features helps prevent any single feature from dominating the regularization term.

**Important Notes:**
- When scaling, it's crucial to follow the practice of fitting the scaler to the training data and then separately transforming both the training and test sets.
- This separation ensures that the scaling is based solely on the training data's statistics, preventing data leakage from the test set.

For a deeper understanding of scaling and its importance in regularized models, refer to the theory videos. They also provide insights into why fitting to the training data and transforming both sets separately is a best practice.


In [12]:
# Scaling the Data

# In this code snippet, we are importing the StandardScaler module from scikit-learn to standardize or scale our dataset.

from sklearn.preprocessing import StandardScaler

# Explanation:
# - Standardization is an essential step when working with regularized models like Ridge and Lasso regression.
# - The StandardScaler is a feature transformation technique provided by scikit-learn.
# - It rescales the features to have a mean of 0 and a standard deviation of 1, making them have similar scales.
# - This ensures that all features contribute equally to regularization, regardless of their original units and magnitudes.

# We'll use the StandardScaler to preprocess our feature data, ensuring that it's appropriately scaled before applying regularization techniques.

# Remember that it's good practice to fit the scaler to the training data and then transform both the training and test sets separately to prevent data leakage.


In [13]:
# help(StandardScaler)

In [14]:
scaler = StandardScaler()

In [15]:
scaler.fit(X_train)

StandardScaler()

In [16]:
X_train = scaler.transform(X_train)

In [17]:
X_test = scaler.transform(X_test)

## Ridge Regression

Make sure to view video lectures for full explanation of Ridge Regression and choosing an alpha.

In [18]:
# Regularization with Ridge Regression

# In this code snippet, we are importing the Ridge module from scikit-learn, which allows us to apply Ridge Regression for regularization.

from sklearn.linear_model import Ridge

# Explanation:
# - Ridge Regression is a regularization technique that adds a penalty term to the linear regression objective function.
# - This penalty term encourages smaller coefficients for features, helping to prevent overfitting and reducing the impact of collinearity (high feature correlation).
# - The Ridge module provided by scikit-learn offers an implementation of Ridge Regression.
# - We'll use this module to create a Ridge Regression model that considers regularization while making predictions.

# By using Ridge Regression, we aim to build a more robust and generalized model, especially when dealing with datasets with multiple features.


In [19]:
ridge_model = Ridge(alpha=10)

In [20]:
# Configuring Ridge Regression Model

# In this code snippet, we are configuring a Ridge Regression model with a specified alpha (λ) parameter.

ridge_model = Ridge(alpha=10)

# Explanation:
# - Ridge Regression introduces a regularization term controlled by the alpha (λ) parameter.
# - The alpha parameter determines the strength of the regularization: higher values of alpha result in stronger regularization.
# - By setting alpha to 10, we are specifying a relatively strong regularization for the Ridge model.
# - A higher alpha value will lead to more aggressive shrinking of feature coefficients, reducing the risk of overfitting.
# - The choice of alpha should be based on model performance evaluation, as it balances the trade-off between model complexity and fit to the data.

# This code initializes our Ridge Regression model with the chosen alpha value, setting the level of regularization to be applied.
``


Ridge(alpha=10)

In [21]:
test_predictions = ridge_model.predict(X_test)

In [22]:
from sklearn.metrics import mean_absolute_error,mean_squared_error

In [23]:
MAE = mean_absolute_error(y_test,test_predictions)
MSE = mean_squared_error(y_test,test_predictions)
RMSE = np.sqrt(MSE)

In [24]:
MAE

0.5774404204714167

In [25]:
RMSE

0.894638646131965

How did it perform on the training set? (This will be used later on for comparison)

In [26]:
# Training Set Performance
train_predictions = ridge_model.predict(X_train)
MAE = mean_absolute_error(y_train,train_predictions)
MAE

0.5288348183025304

### Choosing an alpha value with Cross-Validation

Review the theory video for full details.

In [27]:
# Ridge Regression with Cross-Validated Alpha Selection

# In this code snippet, we are importing the RidgeCV module from scikit-learn, which allows us to perform Ridge Regression with cross-validated alpha selection.

from sklearn.linear_model import RidgeCV

# Explanation:
# - RidgeCV is a variant of Ridge Regression that includes built-in cross-validation for alpha (λ) selection.
# - Cross-validation helps us choose the optimal alpha value by assessing model performance on different alpha values using cross-validation folds.
# - This process avoids the need for manual tuning and ensures the selection of an alpha that balances bias and variance effectively.
# - By using RidgeCV, we streamline the process of selecting the best alpha for Ridge Regression.

# RidgeCV is a convenient choice when you want to automate the alpha selection process and ensure that your regularization parameter is well-suited to your data.


In [28]:
# help(RidgeCV)

In [29]:
# Configuring RidgeCV with Scoring

# In this code snippet, we are configuring a RidgeCV model with specified alpha values and a scoring metric for cross-validation.

# Choosing a scoring: https://scikit-learn.org/stable/modules/model_evaluation.html
# Negative RMSE so all metrics follow convention "Higher is better"

# See all options: sklearn.metrics.SCORERS.keys()

ridge_cv_model = RidgeCV(alphas=(0.1, 1.0, 10.0), scoring='neg_mean_absolute_error')

# Explanation:
# - RidgeCV is being configured to perform Ridge Regression with cross-validated alpha selection.
# - The 'alphas' parameter specifies a set of alpha values to consider during cross-validation.
# - The 'scoring' parameter specifies the evaluation metric to use during cross-validation.
# - In this case, 'neg_mean_absolute_error' is chosen as the scoring metric.
# - It's important to note that for scoring, we use the negative mean absolute error (MAE).
#   - This is a common practice in scikit-learn where higher values of the score indicate better performance.
#   - By using negative MAE, we align the scoring convention with the "higher is better" convention.
# - The specified alpha values (0.1, 1.0, and 10.0) represent different regularization strengths to be evaluated during cross-validation.

# RidgeCV will automatically select the alpha that yields the best performance on the chosen scoring metric, simplifying the alpha selection process.


In [30]:
# The more alpha options you pass, the longer this will take.
# Fortunately our data set is still pretty small
ridge_cv_model.fit(X_train,y_train)

RidgeCV(alphas=array([ 0.1,  1. , 10. ]), scoring='neg_mean_absolute_error')

In [31]:
ridge_cv_model.alpha_

0.1

In [32]:
test_predictions = ridge_cv_model.predict(X_test)

In [33]:
MAE = mean_absolute_error(y_test,test_predictions)
MSE = mean_squared_error(y_test,test_predictions)
RMSE = np.sqrt(MSE)

In [34]:
MAE

0.4273774884351013

In [35]:
RMSE

0.6180719926948981

In [36]:
# Training Set Performance
# Training Set Performance
train_predictions = ridge_cv_model.predict(X_train)
MAE = mean_absolute_error(y_train,train_predictions)
MAE

0.30941321056569865

In [37]:
ridge_cv_model.coef_

array([ 5.40769392,  0.5885865 ,  0.40390395, -6.18263924,  4.59607939,
       -1.18789654, -1.15200458,  0.57837796, -0.1261586 ,  2.5569777 ,
       -1.38900471,  0.86059434,  0.72219553, -0.26129256,  0.17870787,
        0.44353612, -0.21362436, -0.04622473, -0.06441449])


-----

## Lasso Regression

In [1]:
from sklearn.linear_model import LassoCV

In [39]:
# Lasso Regression with Cross-Validated Alpha Selection

# In this code snippet, we are configuring a LassoCV model with parameters for alpha (λ) selection using cross-validation.

# https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html

lasso_cv_model = LassoCV(eps=0.1, n_alphas=100, cv=5)

# Explanation:
# - LassoCV is being configured to perform Lasso Regression with cross-validated alpha selection.
# - The 'eps' parameter specifies the length of the path of alphas to consider.
# - The 'n_alphas' parameter determines the number of alpha values to include in the path.
# - The 'cv' parameter sets the number of cross-validation folds to use for evaluating model performance.
#   - In this case, 'cv=5' indicates 5-fold cross-validation.

# LassoCV will automatically perform cross-validation on different alpha values to find the one that optimizes model performance.
# The 'eps' and 'n_alphas' parameters control the range and granularity of alpha values to explore during the search.

# LassoCV is a useful tool for selecting the optimal alpha for Lasso Regression while considering cross-validated performance.


In [40]:
lasso_cv_model.fit(X_train,y_train)

LassoCV(cv=5, eps=0.1)

In [41]:
lasso_cv_model.alpha_

0.4943070909225832

In [42]:
test_predictions = lasso_cv_model.predict(X_test)

In [43]:
MAE = mean_absolute_error(y_test,test_predictions)
MSE = mean_squared_error(y_test,test_predictions)
RMSE = np.sqrt(MSE)

In [44]:
MAE

0.6541723161252867

In [45]:
RMSE

1.1308001022762548

In [46]:
# Training Set Performance
# Training Set Performance
train_predictions = lasso_cv_model.predict(X_train)
MAE = mean_absolute_error(y_train,train_predictions)
MAE

0.6912807140820709

In [47]:
lasso_cv_model.coef_

array([1.002651  , 0.        , 0.        , 0.        , 3.79745279,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        ])

## Elastic Net

Elastic Net combines the penalties of ridge regression and lasso in an attempt to get the best of both worlds!

In [48]:
from sklearn.linear_model import ElasticNetCV

In [51]:
elastic_model = ElasticNetCV(l1_ratio=[.1, .5, .7,.9, .95, .99, 1],tol=0.01)

In [52]:
elastic_model.fit(X_train,y_train)

ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1], tol=0.01)

In [55]:
elastic_model.l1_ratio_

1.0

In [56]:
test_predictions = elastic_model.predict(X_test)

In [57]:
MAE = mean_absolute_error(y_test,test_predictions)
MSE = mean_squared_error(y_test,test_predictions)
RMSE = np.sqrt(MSE)

In [58]:
MAE

0.5663262117569452

In [59]:
RMSE

0.7485546215633726

In [60]:
# Training Set Performance
# Training Set Performance
train_predictions = elastic_model.predict(X_train)
MAE = mean_absolute_error(y_train,train_predictions)
MAE

0.43075829904723684

In [61]:
elastic_model.coef_

array([ 3.78993643,  0.89232919,  0.28765395, -1.01843566,  2.15516144,
       -0.3567547 , -0.271502  ,  0.09741081,  0.        , -1.05563151,
        0.2362506 ,  0.07980911,  1.26170778,  0.01464706,  0.00462336,
       -0.39986069,  0.        ,  0.        , -0.05343757])

-----
---